# 🔏 Trust, but Verify: Signed Agent Cards

**DevFest Mauritius 2026 · Hands-on lab (about 6 minutes)**

AI agents find each other through **Agent Cards**: small JSON "business cards" that say who an agent is and where to send work. Anyone can write one. Today you'll:

1. **Break the seal**: change one line of a signed card and watch the check fail.
2. **Be the attacker**: forge a card, seal it with your *own* key, and see whether the check notices.
3. **Stretch (fast finishers):** fix the check.

**How to use this notebook:** `File → Save a copy in Drive`, then run each cell with **Shift + Enter**. Lines you should edit are marked `# ✏️ CHANGE THIS LINE`. Every exercise ends with a ✅ or ❌ so you know where you stand.

No API key needed, and nothing leaves your browser session: the "internet" here is a Python dictionary.

## 0 · Setup

Installs the official A2A Python SDK. If Colab asks you to **restart the session**, click *Restart*, then run this cell again.

In [1]:
%pip install -q "a2a-sdk[signing]==1.1.5" "pyjwt[crypto]==2.15.0"

Now using Node v20.19.4 (npm 10.8.2) ~/.local/share/nvm/v20.19.4/bin/node
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mysql-connector-python 8.2.0 requires protobuf<=4.21.12,>=4.21.1, but you have protobuf 7.36.2 which is incompatible.
google-cloud-language 2.12.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: /usr/local/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


This cell prepares everything: the real translator **Lalang**, its key pair, its sealed card, and a pretend internet where public keys are published. You don't need to read it, just run it.

In [2]:
import copy
import json

from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill
from a2a.utils.signing import (
    SignatureVerificationError,
    create_agent_card_signer,
    create_signature_verifier,
)
from cryptography.hazmat.primitives.asymmetric import ec
from google.protobuf.json_format import MessageToDict, ParseDict
from jwt import PyJWK
from jwt.algorithms import ECAlgorithm
from jwt.exceptions import PyJWKError

REAL_URL = "https://lalang.example/a2a"
ATTACKER_URL = "https://lalang-translator.example/a2a"
REAL_JKU = "https://lalang.example/.well-known/jwks.json"
ATTACKER_JKU = "https://lalang-translator.example/.well-known/jwks.json"

INTERNET = {}  # pretend internet: URL -> the public keys published there


def make_key(kid, jku):
    """Create a key pair and publish the public half at `jku`."""
    private_key = ec.generate_private_key(ec.SECP256R1())
    public_key = ECAlgorithm.to_jwk(private_key.public_key(), as_dict=True)
    public_key |= {"kid": kid, "alg": "ES256"}
    INTERNET[jku] = {"keys": [public_key]}
    return private_key, public_key


def make_card(url):
    """Lalang's card. The attacker copies it word for word, except the url."""
    return AgentCard(
        name="Lalang Translator",
        description="Expert English → French translation. Powered by Gemini 3.8 Flash.",
        version="1.0.0",
        supported_interfaces=[
            AgentInterface(url=url, protocol_binding="JSONRPC", protocol_version="1.0")
        ],
        capabilities=AgentCapabilities(),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=[
            AgentSkill(
                id="translate",
                name="Translate text",
                description="English to French",
                tags=["translate"],
            )
        ],
    )


def seal(card, private_key, kid, jku):
    """Sign the card. The header says which key (kid) and where to get it (jku)."""
    sign = create_agent_card_signer(
        signing_key=private_key,
        protected_header={"alg": "ES256", "typ": "JOSE", "kid": kid, "jku": jku},
    )
    return sign(card)


def download_key(kid, jku):
    """A careless key lookup: fetch the key from wherever the card says."""
    for key in INTERNET.get(jku, {}).get("keys", []):
        if key["kid"] == kid:
            return PyJWK(key)
    raise PyJWKError(f"no key {kid!r} published at {jku}")


def is_valid(card, get_key=download_key):
    """True if at least one signature on the card checks out."""
    verify = create_signature_verifier(key_provider=get_key, algorithms=["ES256"])
    try:
        verify(card)
        return True
    except SignatureVerificationError:
        return False


def show(card):
    print(json.dumps(MessageToDict(card), indent=2, ensure_ascii=False))


LALANG_KEY, LALANG_PUBLIC_KEY = make_key("lalang-2026", REAL_JKU)
lalang_card = seal(make_card(REAL_URL), LALANG_KEY, "lalang-2026", REAL_JKU)

show(lalang_card)
print("\n✅ Setup done. Lalang's sealed card is valid:", is_valid(lalang_card))

{
  "name": "Lalang Translator",
  "description": "Expert English → French translation. Powered by Gemini 3.8 Flash.",
  "supportedInterfaces": [
    {
      "url": "https://lalang.example/a2a",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "translate",
      "name": "Translate text",
      "description": "English to French",
      "tags": [
        "translate"
      ]
    }
  ],
  "signatures": [
    {
      "protected": "eyJhbGciOiJFUzI1NiIsImprdSI6Imh0dHBzOi8vbGFsYW5nLmV4YW1wbGUvLndlbGwta25vd24vandrcy5qc29uIiwia2lkIjoibGFsYW5nLTIwMjYiLCJ0eXAiOiJKT1NFIn0",
      "signature": "Tge5L2hYySkQROpTd3R-pAZY-QH-70Insjvs0Vrv707uIQpgwrHnHYOKSSq-Kk1jG3rAXTg-Q78LSi4BH65Wrg"
    }
  ]
}

✅ Setup done. Lalang's sealed card is valid: True


## 1 · Break the seal (2 min)

An attacker gets hold of Lalang's **sealed** card and changes one line: the address where your data goes. Does the seal notice?

Change the marked line to use `ATTACKER_URL`, then run the cell.

In [3]:
card = copy.deepcopy(lalang_card)

card.supported_interfaces[0].url = REAL_URL  # ✏️ CHANGE THIS LINE: use ATTACKER_URL

if card.supported_interfaces[0].url == REAL_URL:
    print("✏️  Not yet: change REAL_URL to ATTACKER_URL, then run again.")
elif is_valid(card):
    print("🤔 The seal didn't notice. Did you re-run the setup cell?")
else:
    print("✅ Rejected! You changed one line and the seal broke.")

✏️  Not yet: change REAL_URL to ATTACKER_URL, then run again.


<details><summary>Stuck? Reveal answer</summary>

```python
card.supported_interfaces[0].url = ATTACKER_URL
```
</details>

### 1b · Bonus: does formatting matter?

Here's the same card with its fields in **reverse order** and lots of **extra spaces**. Before checking, the SDK puts every card into one standard format (*canonicalization*), so only real changes break the seal. Just run it.

In [4]:
reshuffled = dict(reversed(list(MessageToDict(lalang_card).items())))
text = json.dumps(reshuffled, indent=8)
print(text[:250], "...\n")

same_card = ParseDict(json.loads(text), AgentCard())
if is_valid(same_card):
    print("✅ Still valid: formatting doesn't matter.")

{
        "signatures": [
                {
                        "protected": "eyJhbGciOiJFUzI1NiIsImprdSI6Imh0dHBzOi8vbGFsYW5nLmV4YW1wbGUvLndlbGwta25vd24vandrcy5qc29uIiwia2lkIjoibGFsYW5nLTIwMjYiLCJ0eXAiOiJKT1NFIn0",
                        "signa ...

✅ Still valid: formatting doesn't matter.


## 2 · Be the attacker (3 min)

Breaking Lalang's seal didn't work. So you get your **own** key, publish it on **your** website, and seal a word-for-word copy of Lalang's card that points to **your** address.

The victim runs the **same** `is_valid` check as before.

🤔 **Predict first:** will your fake card pass? Then change the marked line to `ATTACKER_JKU` and run.

In [5]:
ATTACKER_KEY, _ = make_key("attacker-key", ATTACKER_JKU)  # your key, on your site

jku = REAL_JKU  # ✏️ CHANGE THIS LINE: point to YOUR published key, ATTACKER_JKU

fake_card = seal(make_card(ATTACKER_URL), ATTACKER_KEY, "attacker-key", jku)

if jku == REAL_JKU:
    print("❌ Rejected: the check looked for your key on Lalang's site. Not there.")
    print("✏️  Now change jku to ATTACKER_JKU and run again.")
elif is_valid(fake_card):
    print("😱 ACCEPTED! Your forged card passed the check.")
    print("   Why? Put your hand up in the room!")
else:
    print("🤔 Unexpected. Did you run the setup cell?")

❌ Rejected: the check looked for your key on Lalang's site. Not there.
✏️  Now change jku to ATTACKER_JKU and run again.


<details><summary>Stuck? Reveal answer</summary>

```python
jku = ATTACKER_JKU
```
The check downloaded the key from the address **written on the card**, and you wrote the card. The seal is real. It just belongs to the wrong person.
</details>

## 3 · Stretch (fast finishers): fix the check

A seal only helps if you know **whose** seal it is. Write `my_get_key` so it only returns keys you **saved in advance** (*pinning*) and ignores the card's `jku`.

- If `kid` is in `TRUSTED_KEYS`, return `PyJWK(TRUSTED_KEYS[kid])`.
- Otherwise, `raise PyJWKError(...)`.

In [6]:
TRUSTED_KEYS = {"lalang-2026": LALANG_PUBLIC_KEY}  # saved before any card arrived


def my_get_key(kid, jku):
    return download_key(kid, jku)  # ✏️ CHANGE THIS LINE: use TRUSTED_KEYS instead


forged = seal(make_card(ATTACKER_URL), ATTACKER_KEY, "attacker-key", ATTACKER_JKU)
real_ok = is_valid(lalang_card, my_get_key)
fake_ok = is_valid(forged, my_get_key)

print("Real card:", "✅ accepted" if real_ok else "❌ rejected (it should be accepted)")
print("Fake card:", "✅ rejected" if not fake_ok else "❌ accepted (still fooled)")
if real_ok and not fake_ok:
    print("\n🎉 Fixed! You check the seal against a key you already trust.")

Real card: ✅ accepted
Fake card: ❌ accepted (still fooled)


<details><summary>Stuck? Reveal answer</summary>

```python
def my_get_key(kid, jku):
    if kid not in TRUSTED_KEYS:
        raise PyJWKError(f"untrusted key: {kid}")
    return PyJWK(TRUSTED_KEYS[kid])  # jku is ignored on purpose
```
</details>

## What you just learned

- **A seal catches tampering.** Change one character and the check fails.
- **The key's source decides trust.** If the card tells you where to get the key, the attacker picks the key.
- **Pin keys you already trust**, require a signature, and check cards in code *before* your AI ever reads them.